<h1><center> IT HELP DESK ANALYTICS AND SLA BREACH PREDICTION </center></h1>

#### Import Required Libraries

In [481]:
import pandas as pd
import numpy as np      
import joblib
from  sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import recall_score, precision_score, f1_score
from pathlib import Path
from sklearn.linear_model import LogisticRegression

#### Install Required Packages

In [437]:
%pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


#### Load the Dataset

In [438]:
df=pd.read_excel(r"C:\Users\ELCOT\Desktop\ITHelpDesk_SLA Breach_Prediction\Excel\Source\Cleaned_Data\ITDesk_SLA_Anaysis.xlsx",sheet_name="Cleaned_Data")

#### Prepare the Dataset
##### To display 1515 rows and 26 columns 

In [439]:
df = df.iloc[:1515].copy()
# Selects the first 1,515 rows and creates a separate copy.

pd.set_option('display.max_rows', None)
# Displays all rows without hiding any rows.

pd.set_option('display.max_columns', None)
# Displays all columns without hiding any columns.

# display(df)
# this line for diplaying 1515 rows
df.head(3)

,Status,Ticket ID,Priority,Source,Topic,Agent Group,Agent Name,Created time,Expected SLA to resolve,Expected SLA to first response,First response time,SLA For first response,Resolution time,SLA For Resolution,Close time,Agent interactions,Survey results,Product group,Support Level,Country,Response Duration (min),Resolution Duration (min),Response SLA Difference (min),Resolution SLA Difference (min),Resolution SLA Validation,Response SLA Validation
0,Closed,1,High,Email,General Inquiry,Security,Venkatesh Rao,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00,2024-07-04 13:02:00,Met,2024-07-04 14:30:00,Met,2024-07-04 14:32:00,5.0,Neutral,Cloud,L3,Oman,20,108,-10.0,-12.0,Timestamp present,Timestamp present
1,Closed,2,High,Chat,Network Issue,Customer Service,Dinesh Pillai,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00,2024-05-23 20:25:00,Met,2024-05-23 22:00:00,Met,2024-05-23 22:05:00,4.0,Dissatisfied,Cloud,L2,Qatar,22,117,-8.0,-3.0,Timestamp present,Timestamp present
2,In Progress,3,Low,Phone,General Inquiry,Development,Rajesh Iyer,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00,2024-04-13 21:41:00,Met,2024-04-14 00:57:00,Not Met,2024-04-14 00:57:00,3.0,Dissatisfied,Software,L1,Bahrain,50,246,-10.0,6.0,Timestamp present,Timestamp present


##### Dataset Dimensions

In [440]:
# to find how many rows and columns
df.shape

(1515, 26)

##### Dataset Structure and Data Types

In [441]:
# Display dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1515 entries, 0 to 1514
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Status                           1515 non-null   str           
 1   Ticket ID                        1515 non-null   object        
 2   Priority                         1515 non-null   str           
 3   Source                           1515 non-null   str           
 4   Topic                            1515 non-null   str           
 5   Agent Group                      1515 non-null   str           
 6   Agent Name                       1515 non-null   str           
 7   Created time                     1515 non-null   datetime64[us]
 8   Expected SLA to resolve          1515 non-null   datetime64[us]
 9   Expected SLA to first response   1515 non-null   datetime64[us]
 10  First response time              1511 non-null   datetime64[us]
 11  SL

##### Data Quality Check

#### Check Missing values

In [442]:
# Count missing values in each column
df.isnull().sum()

Status                             0
Ticket ID                          0
Priority                           0
Source                             0
Topic                              0
Agent Group                        0
Agent Name                         0
Created time                       0
Expected SLA to resolve            0
Expected SLA to first response     0
First response time                4
SLA For first response             0
Resolution time                    3
SLA For Resolution                 0
Close time                         0
Agent interactions                 4
Survey results                     0
Product group                      0
Support Level                      0
Country                            0
Response Duration (min)            4
Resolution Duration (min)          3
Response SLA Difference (min)      4
Resolution SLA Difference (min)    3
Resolution SLA Validation          0
Response SLA Validation            0
dtype: int64

#### Check Duplicate Records

In [443]:
# Check duplicate rows
df.duplicated().sum()

np.int64(0)

##### Check the target variable for Machine Learning

###### Explore the SLA Target

In [444]:
# Check resolution SLA outcome counts
df["SLA For Resolution"].value_counts()

SLA For Resolution
Met        1492
Not Met      23
Name: count, dtype: int64

In [445]:
# Check the unique outcome labels
df["SLA For Resolution"].unique()

<ArrowStringArray>
['Met', 'Not Met']
Length: 2, dtype: str

##### Features and Target selection

In [446]:
# Features
X =  df[
    ["Priority",
     "Source",
     "Topic",
     "Agent Group",
     "Created time",
     "Expected SLA to resolve",
     "Expected SLA to first response",
     "Product group",
     "Support Level",
     "Country"
     ]
].copy()

# Target
y = df["SLA For Resolution"]
y.value_counts()

SLA For Resolution
Met        1492
Not Met      23
Name: count, dtype: int64

In [447]:
# To view the selected features and target
pd.concat([X,y],axis=1)

,Priority,Source,Topic,Agent Group,Created time,Expected SLA to resolve,Expected SLA to first response,Product group,Support Level,Country,SLA For Resolution
0,High,Email,General Inquiry,Security,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00,Cloud,L3,Oman,Met
1,High,Chat,Network Issue,Customer Service,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00,Cloud,L2,Qatar,Met
2,Low,Phone,General Inquiry,Development,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00,Software,L1,Bahrain,Not Met
3,Critical,Chat,Access Request,Development,2024-05-13 12:50:00,2024-05-13 13:50:00,2024-05-13 13:00:00,Network,L2,Kuwait,Met
4,Critical,Portal,Hardware Failure,Customer Service,2024-06-19 22:51:00,2024-06-19 23:51:00,2024-06-19 23:01:00,Hardware,L3,Qatar,Met
5,Medium,Email,Network Issue,Customer Service,2024-04-19 03:41:00,2024-04-19 06:41:00,2024-04-19 04:26:00,Software,L3,Qatar,Not Met
6,Critical,Portal,Access Request,Security,2024-06-10 07:16:00,2024-06-10 08:16:00,2024-06-10 07:26:00,Software,L2,Bahrain,Met
7,High,Portal,General Inquiry,Network Ops,2024-04-09 09:55:00,2024-04-09 11:55:00,2024-04-09 10:25:00,Cloud,L1,UAE,Met
8,Medium,Chat,Access Request,Network Ops,2024-05-10 23:28:00,2024-05-11 02:28:00,2024-05-11 00:13:00,Hardware,L3,Saudi Arabia,Met
9,Critical,Portal,Network Issue,Customer Service,2024-06-10 06:48:00,2024-06-10 07:48:00,2024-06-10 06:58:00,Network,L2,Kuwait,Met


##### Encode the Target Variable

##### Label Encoder

In [448]:
l_encoder = LabelEncoder()
df["SLA_For_Resolution_Encoded"] = l_encoder.fit_transform(df["SLA For Resolution"])
df["SLA_For_Resolution_Encoded"]

0       0
1       0
2       1
3       0
4       0
5       1
6       0
7       0
8       0
9       0
10      0
11      0
12      0
13      0
14      0
15      0
16      0
17      0
18      0
19      0
20      0
21      0
22      0
23      0
24      0
25      0
26      0
27      0
28      0
29      0
30      0
31      0
32      0
33      0
34      0
35      0
36      0
37      0
38      0
39      0
40      0
41      0
42      0
43      0
44      0
45      0
46      0
47      0
48      0
49      0
50      0
51      0
52      0
53      0
54      0
55      0
56      0
57      0
58      0
59      0
60      1
61      0
62      0
63      0
64      0
65      0
66      0
67      0
68      0
69      0
70      0
71      0
72      0
73      0
74      0
75      0
76      0
77      0
78      0
79      0
80      0
81      0
82      0
83      0
84      0
85      0
86      0
87      0
88      0
89      1
90      0
91      0
92      0
93      0
94      0
95      0
96      0
97      0
98      0
99      0


In [449]:
display(df[[
    "SLA For Resolution",
    "SLA_For_Resolution_Encoded"
]].head(5))

,SLA For Resolution,SLA_For_Resolution_Encoded
0,Met,0
1,Met,0
2,Not Met,1
3,Met,0
4,Met,0


In [450]:
dict(zip(
    l_encoder.classes_,
    l_encoder.transform(l_encoder.classes_)
))

{'Met': np.int64(0), 'Not Met': np.int64(1)}

In [451]:
# View all selected features and encoded target column
display(
    pd.concat(
        [X, df["SLA_For_Resolution_Encoded"]],
        axis=1
    ).head(4)
)

# without the head(4) see full(1515) records

,Priority,Source,Topic,Agent Group,Created time,Expected SLA to resolve,Expected SLA to first response,Product group,Support Level,Country,SLA_For_Resolution_Encoded
0,High,Email,General Inquiry,Security,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00,Cloud,L3,Oman,0
1,High,Chat,Network Issue,Customer Service,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00,Cloud,L2,Qatar,0
2,Low,Phone,General Inquiry,Development,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00,Software,L1,Bahrain,1
3,Critical,Chat,Access Request,Development,2024-05-13 12:50:00,2024-05-13 13:50:00,2024-05-13 13:00:00,Network,L2,Kuwait,0


In [452]:
# Data types of X (features)
X.dtypes

Priority                                     str
Source                                       str
Topic                                        str
Agent Group                                  str
Created time                      datetime64[us]
Expected SLA to resolve           datetime64[us]
Expected SLA to first response    datetime64[us]
Product group                                str
Support Level                                str
Country                                      str
dtype: object

##### X now has 10 features: 7 categorical columns and 3 datetime columns.
#### Convert the datetime Features

In [453]:
# Convert datetime columns
datetime_columns = [
    "Created time",
    "Expected SLA to resolve",
    "Expected SLA to first response"
]

for col in datetime_columns:
    X[col] = pd.to_datetime(df[col], errors="coerce")

# View all records
# display(df[datetime_columns])

df[datetime_columns].head(3)

,Created time,Expected SLA to resolve,Expected SLA to first response
0,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00
1,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00
2,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00


##### Create Time-Based Features

In [454]:
# Extract time-based features
X["hour"] = X["Created time"].dt.hour
X["dayofweek"] = X["Created time"].dt.dayofweek
X["is_weekend"] = (X["dayofweek"] >= 5).astype(int)
X["is_business_hours"] = X["hour"].between(9, 18).astype(int)
X["month"] = X["Created time"].dt.month

# Display all records with the new features
X.head(3)

# for full records
# display(X)

,Priority,Source,Topic,Agent Group,Created time,Expected SLA to resolve,Expected SLA to first response,Product group,Support Level,Country,hour,dayofweek,is_weekend,is_business_hours,month
0,High,Email,General Inquiry,Security,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00,Cloud,L3,Oman,12,3,0,1,7
1,High,Chat,Network Issue,Customer Service,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00,Cloud,L2,Qatar,20,3,0,0,5
2,Low,Phone,General Inquiry,Development,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00,Software,L1,Bahrain,20,5,1,0,4


##### Calculate SLA durations

In [455]:
# Calculate expected resolution SLA in hours
X["Resolution_SLA_Hours"] = (
    X["Expected SLA to resolve"] - X["Created time"]
).dt.total_seconds() / 3600

# Calculate expected first-response SLA in minutes
X["First_Response_SLA_Minutes"] = (
    X["Expected SLA to first response"] - X["Created time"]
).dt.total_seconds() / 60

# View first 3 records
X.head(3)

# to view all records
# display(X)



,Priority,Source,Topic,Agent Group,Created time,Expected SLA to resolve,Expected SLA to first response,Product group,Support Level,Country,hour,dayofweek,is_weekend,is_business_hours,month,Resolution_SLA_Hours,First_Response_SLA_Minutes
0,High,Email,General Inquiry,Security,2024-07-04 12:42:00,2024-07-04 14:42:00,2024-07-04 13:12:00,Cloud,L3,Oman,12,3,0,1,7,2.0,30.0
1,High,Chat,Network Issue,Customer Service,2024-05-23 20:03:00,2024-05-23 22:03:00,2024-05-23 20:33:00,Cloud,L2,Qatar,20,3,0,0,5,2.0,30.0
2,Low,Phone,General Inquiry,Development,2024-04-13 20:51:00,2024-04-14 00:51:00,2024-04-13 21:51:00,Software,L1,Bahrain,20,5,1,0,4,4.0,60.0


In [456]:
# To check  above columns
X.shape

(1515, 17)

##### Remove the raw datetime columns from X (or)
##### Remove Unprocessed Datetime Columns

In [457]:
# Remove raw datetime columns from X
X = X.drop(
    columns=[
        "Created time",
        "Expected SLA to resolve",
        "Expected SLA to first response"
    ]
)

# Display the first 3 records
X.head(3)

,Priority,Source,Topic,Agent Group,Product group,Support Level,Country,hour,dayofweek,is_weekend,is_business_hours,month,Resolution_SLA_Hours,First_Response_SLA_Minutes
0,High,Email,General Inquiry,Security,Cloud,L3,Oman,12,3,0,1,7,2.0,30.0
1,High,Chat,Network Issue,Customer Service,Cloud,L2,Qatar,20,3,0,0,5,2.0,30.0
2,Low,Phone,General Inquiry,Development,Software,L1,Bahrain,20,5,1,0,4,4.0,60.0


In [458]:
# Check missing values in X
X.isnull().sum()

Priority                      0
Source                        0
Topic                         0
Agent Group                   0
Product group                 0
Support Level                 0
Country                       0
hour                          0
dayofweek                     0
is_weekend                    0
is_business_hours             0
month                         0
Resolution_SLA_Hours          0
First_Response_SLA_Minutes    0
dtype: int64

##### Prepare Final Features and Target

In [459]:
# Assign the already encoded target
y = df["SLA_For_Resolution_Encoded"]

# View all selected features and encoded target
final_data = pd.concat(
    [X, y.rename("SLA_For_Resolution_Encoded")],
    axis=1
)

# Display first 5 records
display(final_data.head())

# Display dataset shape
print("Final dataset shape:", final_data.shape)

,Priority,Source,Topic,Agent Group,Product group,Support Level,Country,hour,dayofweek,is_weekend,is_business_hours,month,Resolution_SLA_Hours,First_Response_SLA_Minutes,SLA_For_Resolution_Encoded
0,High,Email,General Inquiry,Security,Cloud,L3,Oman,12,3,0,1,7,2.0,30.0,0
1,High,Chat,Network Issue,Customer Service,Cloud,L2,Qatar,20,3,0,0,5,2.0,30.0,0
2,Low,Phone,General Inquiry,Development,Software,L1,Bahrain,20,5,1,0,4,4.0,60.0,1
3,Critical,Chat,Access Request,Development,Network,L2,Kuwait,12,0,0,1,5,1.0,10.0,0
4,Critical,Portal,Hardware Failure,Customer Service,Hardware,L3,Qatar,22,2,0,0,6,1.0,10.0,0


Final dataset shape: (1515, 15)


##### Identify the categorical columns

In [460]:
# Identify categorical feature columns
categorical_columns = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['Priority', 'Source', 'Topic', 'Agent Group', 'Product group', 'Support Level', 'Country']


#### Encode Categorical Features

In [461]:
# Encode categorical features
X_encoded = pd.get_dummies(X, dtype=int)

# View first 5 records
display(X_encoded.head())

# Check shape
print("Encoded feature shape:", X_encoded.shape)

,hour,dayofweek,is_weekend,is_business_hours,month,Resolution_SLA_Hours,First_Response_SLA_Minutes,Priority_Critical,Priority_High,Priority_Low,Priority_Medium,Source_Chat,Source_Email,Source_Phone,Source_Portal,Topic_Access Request,Topic_General Inquiry,Topic_Hardware Failure,Topic_Network Issue,Topic_Software Bug,Agent Group_Customer Service,Agent Group_Development,Agent Group_IT Support,Agent Group_Network Ops,Agent Group_Security,Product group_Cloud,Product group_Hardware,Product group_Network,Product group_Software,Support Level_L1,Support Level_L2,Support Level_L3,Country_Bahrain,Country_India,Country_Kuwait,Country_Oman,Country_Qatar,Country_Saudi Arabia,Country_UAE,Country_Unknown
0,12,3,0,1,7,2.0,30.0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,20,3,0,0,5,2.0,30.0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2,20,5,1,0,4,4.0,60.0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0
3,12,0,0,1,5,1.0,10.0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,0
4,22,2,0,0,6,1.0,10.0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0


Encoded feature shape: (1515, 40)


### Check the encoded dataset

In [462]:
print("Number of records:", X_encoded.shape[0])
print("Number of features:", X_encoded.shape[1])

print("\nTarget distribution:")
print(y.value_counts())

Number of records: 1515
Number of features: 40

Target distribution:
SLA_For_Resolution_Encoded
0    1492
1      23
Name: count, dtype: int64


In [463]:
final_encoded_data = X_encoded.copy()
final_encoded_data["SLA_For_Resolution_Encoded"] = y

#  for view first 3 records
display(final_encoded_data.head(3))

# for all data
# display(final_encoded_data)

final_encoded_data.shape

,hour,dayofweek,is_weekend,is_business_hours,month,Resolution_SLA_Hours,First_Response_SLA_Minutes,Priority_Critical,Priority_High,Priority_Low,Priority_Medium,Source_Chat,Source_Email,Source_Phone,Source_Portal,Topic_Access Request,Topic_General Inquiry,Topic_Hardware Failure,Topic_Network Issue,Topic_Software Bug,Agent Group_Customer Service,Agent Group_Development,Agent Group_IT Support,Agent Group_Network Ops,Agent Group_Security,Product group_Cloud,Product group_Hardware,Product group_Network,Product group_Software,Support Level_L1,Support Level_L2,Support Level_L3,Country_Bahrain,Country_India,Country_Kuwait,Country_Oman,Country_Qatar,Country_Saudi Arabia,Country_UAE,Country_Unknown,SLA_For_Resolution_Encoded
0,12,3,0,1,7,2.0,30.0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0
1,20,3,0,0,5,2.0,30.0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
2,20,5,1,0,4,4.0,60.0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,1


(1515, 41)

In [464]:
# this for feature only
X_encoded.head(3)
# X_encoded.shape

,hour,dayofweek,is_weekend,is_business_hours,month,Resolution_SLA_Hours,First_Response_SLA_Minutes,Priority_Critical,Priority_High,Priority_Low,Priority_Medium,Source_Chat,Source_Email,Source_Phone,Source_Portal,Topic_Access Request,Topic_General Inquiry,Topic_Hardware Failure,Topic_Network Issue,Topic_Software Bug,Agent Group_Customer Service,Agent Group_Development,Agent Group_IT Support,Agent Group_Network Ops,Agent Group_Security,Product group_Cloud,Product group_Hardware,Product group_Network,Product group_Software,Support Level_L1,Support Level_L2,Support Level_L3,Country_Bahrain,Country_India,Country_Kuwait,Country_Oman,Country_Qatar,Country_Saudi Arabia,Country_UAE,Country_Unknown
0,12,3,0,1,7,2.0,30.0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0
1,20,3,0,0,5,2.0,30.0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0
2,20,5,1,0,4,4.0,60.0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0


### train test split

In [465]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)


# To view the shape of train and test sets

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1060, 40)
X_test: (455, 40)
y_train: (1060,)
y_test: (455,)


### Train the model

In [466]:

model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced_subsample",
    random_state=42
)

model.fit(X_train, y_train)

print("Improved Random Forest model trained successfully!")


Improved Random Forest model trained successfully!


### Predict the SLA status
###### On test model

In [467]:

y_pred = model.predict(X_test)

print("Predictions completed successfully!")
print("First 10 predictions:", y_pred[:10])

Predictions completed successfully!
First 10 predictions: [0 0 0 0 0 0 0 0 0 0]


### Evaluate the model

In [468]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Met", "Not Met"],
        zero_division=0
    )
)


Confusion Matrix:
[[448   0]
 [  7   0]]

Classification Report:
              precision    recall  f1-score   support

         Met       0.98      1.00      0.99       448
     Not Met       0.00      0.00      0.00         7

    accuracy                           0.98       455
   macro avg       0.49      0.50      0.50       455
weighted avg       0.97      0.98      0.98       455



In [469]:
# the model's breach-detection performance is not good enough yet in the above two cells.

# output shows:
# - ✅ Confusion matrix and classification report are generated correctly.
# - ✅ All 448 Met tickets are identified correctly.
# - ❌ 0 out of 7 Not Met tickets are detected.
# - ❌ Not Met recall is 0%.

#### Check the class distribution

In [470]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training target distribution:
SLA_For_Resolution_Encoded
0    1044
1      16
Name: count, dtype: int64

Testing target distribution:
SLA_For_Resolution_Encoded
0    448
1      7
Name: count, dtype: int64


In [471]:
# Balance the training data only
ros = RandomOverSampler(random_state=42)

X_train_resampled, y_train_resampled = ros.fit_resample(
    X_train, y_train
)

print("Before oversampling:")
print(y_train.value_counts())

print("\nAfter oversampling:")
print(y_train_resampled.value_counts())

Before oversampling:
SLA_For_Resolution_Encoded
0    1044
1      16
Name: count, dtype: int64

After oversampling:
SLA_For_Resolution_Encoded
0    1044
1    1044
Name: count, dtype: int64


##### Train a separate Random Forest model

In [472]:
ros_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

ros_model.fit(X_train_resampled, y_train_resampled)

print("Random Forest model trained on oversampled data!")

Random Forest model trained on oversampled data!


In [473]:
y_pred_ros = ros_model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_ros, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_ros,
    labels=[0, 1],
    target_names=["Met", "Not Met"],
    zero_division=0
))

Confusion Matrix:
[[448   0]
 [  7   0]]

Classification Report:
              precision    recall  f1-score   support

         Met       0.98      1.00      0.99       448
     Not Met       0.00      0.00      0.00         7

    accuracy                           0.98       455
   macro avg       0.49      0.50      0.50       455
weighted avg       0.97      0.98      0.98       455



In [474]:
y_prob_ros = ros_model.predict_proba(X_test)[:, 1]

print("Minimum breach probability:", y_prob_ros.min())
print("Maximum breach probability:", y_prob_ros.max())
print("\nFirst 20 breach probabilities:")
print(y_prob_ros[:20])

Minimum breach probability: 0.0
Maximum breach probability: 0.26

First 20 breach probabilities:
[0.00333333 0.03333333 0.00666667 0.01333333 0.03       0.04333333
 0.00666667 0.01333333 0.15666667 0.00333333 0.01666667 0.00333333
 0.01666667 0.01666667 0.         0.04       0.00666667 0.
 0.00666667 0.00333333]


In [475]:
threshold_ros = 0.10

y_pred_ros_10 = (y_prob_ros >= threshold_ros).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_ros_10, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_ros_10,
    labels=[0, 1],
    target_names=["Met", "Not Met"],
    zero_division=0
))

Confusion Matrix:
[[430  18]
 [  7   0]]

Classification Report:
              precision    recall  f1-score   support

         Met       0.98      0.96      0.97       448
     Not Met       0.00      0.00      0.00         7

    accuracy                           0.95       455
   macro avg       0.49      0.48      0.49       455
weighted avg       0.97      0.95      0.96       455



In [476]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression predictions created!")

Logistic Regression predictions created!


In [477]:
models = {
    "Logistic Regression": y_pred_lr,
    "Random Forest": y_pred,
    "Oversampled Random Forest": y_pred_ros
}

for name, predictions in models.items():
    print(f"\n{name}")
    print("Not Met Recall:",
          round(recall_score(y_test, predictions, pos_label=1, zero_division=0), 3))
    print("Not Met Precision:",
          round(precision_score(y_test, predictions, pos_label=1, zero_division=0), 3))
    print("Not Met F1-score:",
          round(f1_score(y_test, predictions, pos_label=1, zero_division=0), 3))


Logistic Regression
Not Met Recall: 0.429
Not Met Precision: 0.025
Not Met F1-score: 0.047

Random Forest
Not Met Recall: 0.0
Not Met Precision: 0.0
Not Met F1-score: 0.0

Oversampled Random Forest
Not Met Recall: 0.0
Not Met Precision: 0.0
Not Met F1-score: 0.0


#### Finalize Logistic Regression evaluation

In [478]:
from sklearn.metrics import confusion_matrix, classification_report

print("Logistic Regression — Final Evaluation")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr,
    labels=[0, 1],
    target_names=["Met", "Not Met"],
    zero_division=0
))

Logistic Regression — Final Evaluation

Confusion Matrix:
[[330 118]
 [  4   3]]

Classification Report:
              precision    recall  f1-score   support

         Met       0.99      0.74      0.84       448
     Not Met       0.02      0.43      0.05         7

    accuracy                           0.73       455
   macro avg       0.51      0.58      0.45       455
weighted avg       0.97      0.73      0.83       455



### Save the final model comparison

In [479]:
final_comparison = pd.DataFrame([
    {
        "Model": name,
        "Not Met Recall": recall_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "Not Met Precision": precision_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "Not Met F1-score": f1_score(
            y_test, predictions, pos_label=1, zero_division=0
        )
    }
    for name, predictions in models.items()
])

display(final_comparison.round(3))

,Model,Not Met Recall,Not Met Precision,Not Met F1-score
0,Logistic Regression,0.429,0.025,0.047
1,Random Forest,0.000,0.000,0.000
2,Oversampled Random Forest,0.000,0.000,0.000


#### Save the trained model

In [480]:
# Create a folder to store the trained model
model_folder = Path("models")
model_folder.mkdir(exist_ok=True)

# Save the trained Logistic Regression model
joblib.dump(lr_model, model_folder / "sla_breach_model.pkl")

# Save the feature column order
joblib.dump(X_train.columns.tolist(), model_folder / "model_features.pkl")

print("Model saved successfully!")
print("Model path:", model_folder / "sla_breach_model.pkl")
print("Feature list saved successfully!")

Model saved successfully!
Model path: models\sla_breach_model.pkl
Feature list saved successfully!


##### Train the Logistic Regression Model

##### Logistic Regression with new threshold value 

In [482]:
# Predict breach probabilities
y_prob_lr = lr_model.predict_proba(X_test)[:, 1]

# Test a lower decision threshold
threshold_lr = 0.30
y_pred_lr_30 = (y_prob_lr >= threshold_lr).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr_30, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr_30,
    labels=[0, 1],
    target_names=["Met", "Not Met"],
    zero_division=0
))

Confusion Matrix:
[[268 180]
 [  2   5]]

Classification Report:
              precision    recall  f1-score   support

         Met       0.99      0.60      0.75       448
     Not Met       0.03      0.71      0.05         7

    accuracy                           0.60       455
   macro avg       0.51      0.66      0.40       455
weighted avg       0.98      0.60      0.74       455



In [484]:
models = {
    "Logistic Regression": y_pred_lr,
    "Logistic Regression (Threshold 0.30)": y_pred_lr_30,
    "Random Forest": y_pred,
    "Oversampled Random Forest": y_pred_ros
}

In [485]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr_30, labels=[0, 1]))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_lr_30,
    labels=[0, 1],
    target_names=["Met", "Not Met"],
    zero_division=0
))

Confusion Matrix:
[[268 180]
 [  2   5]]

Classification Report:
              precision    recall  f1-score   support

         Met       0.99      0.60      0.75       448
     Not Met       0.03      0.71      0.05         7

    accuracy                           0.60       455
   macro avg       0.51      0.66      0.40       455
weighted avg       0.98      0.60      0.74       455



In [486]:
final_comparison = pd.DataFrame([
    {
        "Model": name,
        "Not Met Recall": recall_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "Not Met Precision": precision_score(
            y_test, predictions, pos_label=1, zero_division=0
        ),
        "Not Met F1-score": f1_score(
            y_test, predictions, pos_label=1, zero_division=0
        )
    }
    for name, predictions in models.items()
])

display(final_comparison.round(3))

,Model,Not Met Recall,Not Met Precision,Not Met F1-score
0,Logistic Regression,0.429,0.025,0.047
1,Logistic Regression (Threshold 0.30),0.714,0.027,0.052
2,Random Forest,0.000,0.000,0.000
3,Oversampled Random Forest,0.000,0.000,0.000
